# Sesión 2.2 — Ampliaciones y ejercicios

Esta parte reúne los ejercicios con vuelos y las ampliaciones del notebook original. Es independiente de la Parte 1: ejecuta primero la celda de preparación siguiente.


## Preparación de datos y librerías

Cargamos de nuevo las librerías y la tabla de teléfonos para que esta parte pueda ejecutarse por separado.


In [ ]:
from pathlib import Path  # Construye rutas portables a las carpetas y archivos.
import numpy as np  # Aporta funciones numéricas y herramientas para simulación.
import pandas as pd  # Permite leer y manipular tablas de datos.
import matplotlib.pyplot as plt  # Permite crear y mostrar gráficos.
from scipy import stats  # Contiene distribuciones y pruebas estadísticas.
from IPython.display import display  # Muestra tablas con formato dentro del notebook.

DATA_PATH = Path.cwd().parent.parent / "datos"  # Localiza la carpeta de datos desde la ruta de trabajo.

datos = pd.read_csv(DATA_PATH / 'Mobiles.txt', sep=';', na_values=['NA', ''])  # Lee la tabla de móviles y convierte esos marcadores en ausentes.
display(datos)  # Muestra los datos para inspeccionar sus columnas y filas.

## Comparación de dos medias

### Muestras independientes

Comparamos el importe medio de la factura entre quienes tienen contrato `Plan` y `Prepaid`. Preparamos los dos grupos eliminando las facturas ausentes; después visualizamos sus distribuciones con un diagrama de cajas y un gráfico de puntos. Finalmente, estimamos la diferencia de medias y realizamos contrastes bilateral y unilateral por separado. **El orden importa:** en todos los cálculos la diferencia es la media de `Plan` menos la de `Prepaid`.

Recurso para explorar distribuciones muestrales: http://onlinestatbook.com/stat_sim/sampling_dist/

In [ ]:
grupos = [(str(nombre), parte['Bill'].dropna()) for nombre, parte in datos.groupby('PlanType', sort=True)]  # Agrupa las facturas por contrato y elimina ausentes de cada grupo.
(nombre1, grupo1), (nombre2, grupo2) = grupos  # Separa las etiquetas y los datos de los dos grupos.
print('Orden de la diferencia de medias:', nombre1, '-', nombre2)  # Deja claro el orden usado para restar las medias.
print(f'Tamaños de los grupos: {nombre1} = {len(grupo1)}, {nombre2} = {len(grupo2)}')  # Informa cuántos datos válidos hay en cada grupo.

#### Diagrama de cajas por tipo de contrato

El **boxplot** resume la mediana (línea central), el rango intercuartílico (caja), la extensión habitual de los datos (bigotes) y los valores atípicos (puntos). Permite comparar la posición y la dispersión de las facturas en ambos grupos, pero no muestra todas las observaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico.
ax.boxplot([grupo1, grupo2], tick_labels=[nombre1, nombre2], patch_artist=True,  # Dibuja una caja por grupo con las etiquetas correspondientes.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Establece color y borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Define el estilo de bigotes y extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Destaca la mediana de cada grupo.
ax.set_title('Facturas por tipo de contrato: diagrama de cajas', fontsize=20)  # Añade un título descriptivo.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta el eje con la variable medida.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el diagrama de cajas.

#### Gráfico de puntos (stripchart)

El **stripchart** representa cada factura individual. El pequeño desplazamiento horizontal aleatorio evita que los puntos con importes iguales queden totalmente superpuestos; **no cambia** el valor de la factura ni el grupo de cada estudiante.

In [ ]:
rng_puntos = np.random.default_rng(12345)  # Fija una semilla para que el desplazamiento de puntos sea reproducible.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y el eje para el gráfico.
for posicion, grupo in enumerate((grupo1, grupo2), start=1):  # Recorre los dos grupos y asigna a cada uno una posición horizontal.
    x = posicion + rng_puntos.uniform(-0.12, 0.12, size=len(grupo))  # Añade una pequeña dispersión horizontal para evitar solapamientos.
    ax.scatter(x, grupo, s=65, color='#9fc3e4', edgecolor='black',  # Dibuja una marca por factura con color y borde.
               linewidth=0.8, alpha=0.7)  # Ajusta el grosor del borde y la transparencia de los puntos.
ax.set_xlim(0.5, 2.5)  # Deja espacio a ambos lados de las dos columnas de puntos.
ax.set_xticks([1, 2], [nombre1, nombre2])  # Etiqueta cada posición con el nombre del grupo.
ax.set_title('Facturas por tipo de contrato: gráfico de puntos', fontsize=20)  # Añade un título explicativo.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta el eje vertical con la variable.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas.
plt.show()  # Muestra el gráfico de puntos.

#### IC del 95 % para la diferencia de medias

Estimamos $\mu_{\mathrm{Plan}}-\mu_{\mathrm{Prepaid}}$ con un **intervalo de confianza del 95 %**. Un intervalo completamente positivo indica que las facturas medias del primer grupo son mayores. Usamos la distribución t y una **varianza común** para ambos grupos (`usevar='pooled'`); este método presupone observaciones independientes y varianzas poblacionales iguales. Como la distribución de las facturas es asimétrica, conviene interpretar este modelo con cautela.

In [ ]:
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW  # Importa herramientas para comparar medias y calcular sus intervalos.

comparacion = CompareMeans(DescrStatsW(grupo1), DescrStatsW(grupo2))  # Prepara la comparación usando los dos grupos en el orden establecido.
ic_95 = comparacion.tconfint_diff(alpha=0.05, usevar='pooled')  # Calcula el IC del 95 % suponiendo varianzas poblacionales iguales.
print(f'IC 95 % para la media de {nombre1} − la de {nombre2}: ({ic_95[0]:.2f}, {ic_95[1]:.2f})')  # Muestra los límites de la diferencia de medias.

#### IC del 99 % para la diferencia de medias

Repetimos la estimación con **mayor confianza (99 %)** sobre los mismos grupos y bajo el mismo supuesto de varianzas iguales. Al aumentar la confianza, el intervalo suele ser **más ancho**; compara sus límites con los del IC del 95 %.

In [ ]:
ic_99 = comparacion.tconfint_diff(alpha=0.01, usevar='pooled')  # Calcula el IC del 99 % con el mismo método y los mismos grupos.
print(f'IC 99 % para la media de {nombre1} − la de {nombre2}: ({ic_99[0]:.2f}, {ic_99[1]:.2f})')  # Imprime los límites para compararlos con el IC del 95 %.

#### Contraste t bilateral: ¿son distintas las medias?

La **prueba t de dos muestras independientes** contrasta si las medias poblacionales de las facturas difieren, sin indicar de antemano cuál sería mayor:

$$
\left\{
\begin{aligned}
H_0 &: \mu_{\mathrm{Plan}} = \mu_{\mathrm{Prepaid}} \\
H_1 &: \mu_{\mathrm{Plan}} \ne \mu_{\mathrm{Prepaid}}
\end{aligned}
\right.
$$

El estadístico t compara la diferencia observada con su error estándar. El **valor p bilateral** mide cuán extrema sería una diferencia de esa magnitud (en cualquiera de las dos direcciones) si $H_0$ fuese cierta. Si $p<0{,}05$, rechazamos la igualdad de medias; si no, no la rechazamos. Esta versión (`equal_var=True`) presupone **independencia y varianzas poblacionales iguales**, además de normalidad aproximada para muestras pequeñas. La asimetría y los valores atípicos vistos antes aconsejan cautela; más adelante se comparan las varianzas.

In [ ]:
resultado_bilateral = stats.ttest_ind(grupo1, grupo2, equal_var=True, alternative='two-sided')  # Contrasta bilateralmente las medias suponiendo varianzas iguales.
print(f'Prueba t bilateral ({nombre1} frente a {nombre2}):')  # Identifica el contraste y el orden de los grupos.
print(f't = {resultado_bilateral.statistic:.4f}; p-valor = {resultado_bilateral.pvalue:.4g}')  # Muestra el estadístico t y su valor p.
if resultado_bilateral.pvalue < 0.05:  # Compara el valor p con el nivel de significación del 5 %.
    print('Se rechaza la igualdad de medias al nivel del 5 %.')  # Informa de evidencia de medias distintas.
else:  # Se ejecuta cuando el valor p es al menos 0,05.
    print('No se rechaza la igualdad de medias al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar la igualdad.

#### Contraste t unilateral: ¿es mayor la media de Plan?

Ahora examinamos una **dirección concreta**. Este contraste debe plantearse **antes de observar los resultados**: la alternativa dice que la factura media de `Plan` supera a la de `Prepaid`.

$$
\left\{
\begin{aligned}
H_0 &: \mu_{\mathrm{Plan}} \le \mu_{\mathrm{Prepaid}} \\
H_1 &: \mu_{\mathrm{Plan}} > \mu_{\mathrm{Prepaid}}
\end{aligned}
\right.
$$

Con los grupos en el orden `Plan`, `Prepaid`, `alternative='greater'` contrasta exactamente esa dirección. Si el **valor p unilateral** es menor que $0{,}05$, rechazamos $H_0$ en favor de que la media de `Plan` es mayor; si no, no tenemos evidencia suficiente para esa afirmación. Mantenemos los **mismos supuestos de independencia y varianzas iguales** que en la prueba bilateral.

In [ ]:
resultado_unilateral = stats.ttest_ind(grupo1, grupo2, equal_var=True, alternative='greater')  # Contrasta si la media del primer grupo supera la del segundo.
print(f'Prueba t unilateral (media de {nombre1} mayor que la de {nombre2}):')  # Describe la dirección de la alternativa.
print(f't = {resultado_unilateral.statistic:.4f}; p-valor = {resultado_unilateral.pvalue:.4g}')  # Muestra estadístico y valor p unilateral.
if resultado_unilateral.pvalue < 0.05:  # Comprueba significación al 5 % en la dirección planteada.
    print(f'Se rechaza H₀: hay evidencia de que la media de {nombre1} es mayor.')  # Expresa la decisión en el contexto de los grupos.
else:  # Se ejecuta si el valor p no alcanza el umbral.
    print(f'No hay evidencia suficiente de que la media de {nombre1} sea mayor.')  # Evita concluir que la media es necesariamente menor.

#### Microejercicio: comparar medias sin suponer varianzas iguales

En el ejemplo se usa Student con varianzas iguales. Repite el intervalo bilateral del 99 % con Welch, que no exige ese supuesto. Completa los argumentos de `tconfint_diff`; en esta función `usevar='unequal'` selecciona Welch. Compara el resultado con `ic_99` y comenta si ambos intervalos llevan a la misma conclusión respecto a una diferencia nula.

In [ ]:
try:
    # Microejercicio: reutiliza la comparación anterior y cambia el supuesto de varianzas.
    ic_welch_99 = comparacion.tconfint_diff(alpha=..., usevar=...)  # Completa alfa para un IC del 99 % y elige Welch.
    print(f'IC Welch del 99 %: ({ic_welch_99[0]:.2f}, {ic_welch_99[1]:.2f})')  # Muestra el intervalo que no presupone varianzas iguales.
    print(f'IC Student del 99 %: ({ic_99[0]:.2f}, {ic_99[1]:.2f})')  # Compara con el IC calculado bajo varianzas iguales.
    # Comprueba qué intervalo contiene 0 y si las conclusiones coinciden.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Datos apareados

En la encuesta de estudiantes, cada fila corresponde a la misma persona en **ambas** mediciones:

- `Sex`: sexo de la persona.
- `HrsSleep`: horas de sueño cada noche.
- `SQpick`: letra elegida entre Q y S.
- `Height`: altura en pulgadas.
- `RandNumb`: número entero entre 1 y 10 elegido al azar.
- `Fastest`: máxima velocidad a la que ha conducido, en millas por hora.
- `RtSpan`: longitud de la palma derecha, en centímetros.
- `LftSpan`: longitud de la palma izquierda, en centímetros.
- `Form`: orden en que se presentaron las opciones Q y S.

#### Lectura de datos y selección de pares completos

`pd.read_csv(..., sep='\t')` lee el fichero separado por tabulaciones; `display` muestra las filas para inspeccionarlas. Seleccionamos las columnas `LftSpan` y `RtSpan` y aplicamos `dropna()` **a ambas a la vez**: si a una persona le falta una medida, excluimos sus dos valores para no romper el emparejamiento. Mostramos cuántos pares completos quedan. Después veremos cada gráfico y cada prueba en una celda distinta.

Descripción de la encuesta: http://sites.stat.psu.edu/~dhunter/250h/data/readme.html

In [ ]:
PennState1 = pd.read_csv(DATA_PATH / 'PennState1.txt', sep='\t')  # Lee la encuesta separada por tabulaciones.
display(PennState1)  # Muestra la tabla para inspeccionar columnas y observaciones.

# Conservamos únicamente a quienes tienen medidas de ambas palmas.
pares = PennState1[['LftSpan', 'RtSpan']].dropna()  # Selecciona ambas medidas a la vez y elimina personas con algún dato ausente.
print(f'Personas con ambas medidas: {len(pares)}; pares incompletos excluidos: {len(PennState1) - len(pares)}')  # Informa los pares válidos y los excluidos.

#### Diagrama de cajas de las longitudes

Comparamos las **distribuciones** de las palmas izquierda y derecha: la línea de cada caja es la mediana, la caja abarca el 50 % central de los valores, los bigotes muestran la extensión habitual y los puntos aislados señalan posibles valores atípicos. Sirve para ver posición y dispersión de cada lado, pero **no permite saber qué dos medidas corresponden a la misma persona**; para eso usamos el gráfico siguiente.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura amplia y sus ejes.
ax.boxplot([pares['LftSpan'], pares['RtSpan']], tick_labels=['Izquierda', 'Derecha'],  # Dibuja una caja por cada lado de la palma.
           patch_artist=True,  # Permite colorear el interior de las cajas.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Establece el relleno y el borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Define los estilos de bigotes y extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Resalta la mediana de cada distribución.
ax.set_title('Longitud de las palmas: diagrama de cajas', fontsize=20)  # Añade el título.
ax.set_ylabel('Palma (cm)', fontsize=16)  # Etiqueta la medida y sus unidades.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
plt.show()  # Muestra el diagrama de cajas.

#### Gráfico de dispersión de las medidas emparejadas

Cada punto representa a **una persona**: su longitud izquierda es la coordenada horizontal y la derecha la vertical. La línea granate es la bisectriz $y=x$: un punto **encima** indica palma derecha mayor; uno **debajo**, palma izquierda mayor; sobre la línea, igualdad. Así vemos a la vez la relación entre ambas medidas y las diferencias dentro de cada par. Los puntos superpuestos pueden corresponder a varias personas; el gráfico no sustituye al contraste.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico de dispersión.
ax.scatter(pares['LftSpan'], pares['RtSpan'], s=65, color='#9fc3e4',  # Dibuja cada persona con izquierda en X y derecha en Y.
           edgecolor='black', linewidth=0.8)  # Añade bordes para diferenciar puntos.
limites = [pares.min().min(), pares.max().max()]  # Obtiene el mínimo y máximo comunes para la línea de referencia.
ax.plot(limites, limites, color='#800000', linewidth=1.8)  # Dibuja y=x, donde ambas medidas son iguales.
ax.set_title('Comparación de palmas por persona', fontsize=20)  # Añade el título.
ax.set_xlabel('Palma izquierda (cm)', fontsize=16)  # Etiqueta el eje X.
ax.set_ylabel('Palma derecha (cm)', fontsize=16)  # Etiqueta el eje Y.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
plt.show()  # Renderiza el gráfico.

#### Prueba t de muestras independientes: ejemplo incorrecto

Para comparar métodos, calculamos lo que daría `stats.ttest_ind` al tratar las mediciones de la mano derecha y de la izquierda como si procediesen de **personas distintas**. Plantearíamos un contraste bilateral de igualdad de medias poblacionales:

$$
\left\{
\begin{aligned}
H_0 &: \mu_{\mathrm{derecha}} = \mu_{\mathrm{izquierda}} \\
H_1 &: \mu_{\mathrm{derecha}} \ne \mu_{\mathrm{izquierda}}
\end{aligned}
\right.
$$

`equal_var=True` añade el supuesto de varianzas poblacionales iguales. **Aquí no se cumple el supuesto de independencia entre grupos:** cada longitud derecha tiene su correspondiente longitud izquierda en la misma persona. Ignorar esta relación calcula un error estándar y un valor p para un diseño distinto; aunque se muestren por motivos didácticos, **no deben usarse para concluir**. Tras los gráficos se presenta la prueba válida.

In [ ]:
# SOLO PARA COMPARAR MÉTODOS: esta prueba no es válida para estas mediciones apareadas.
resultado_independiente = stats.ttest_ind(pares['RtSpan'], pares['LftSpan'], equal_var=True)  # Calcula deliberadamente el resultado incorrecto como si fueran grupos independientes.
print(f'Tratamiento incorrecto como muestras independientes: t = {resultado_independiente.statistic:.4f}; '  # Inicia el mensaje con el estadístico calculado.
      f'p-valor bilateral = {resultado_independiente.pvalue:.4g}')  # Completa el mensaje con el p-valor bilateral.
print('No usar este valor p para sacar conclusiones: las dos medidas pertenecen a las mismas personas.')  # Recuerda por qué no se debe interpretar este contraste.

### Gráficos para datos apareados

La función siguiente prepara **cuatro vistas de los mismos pares completos**: (1) relación entre ambas palmas y recta de igualdad; (2) Bland–Altman, con la diferencia derecha − izquierda frente al promedio de cada par, la diferencia media y límites descriptivos de acuerdo (media ± 1,96 desviaciones típicas de las diferencias); (3) diferencias individuales respecto a cero; (4) perfiles que unen las dos medidas de cada persona. Estos límites de Bland–Altman **no son** un intervalo de confianza para la diferencia media.

En cada perfil hay **dos puntos**, uno para la palma izquierda y otro para la derecha de la **misma persona**, unidos por una línea. El gris tenue indica palmas iguales, el azul oscuro una derecha menor y el azul claro una derecha mayor. La línea granate gruesa une las **medias de cada lado**; no corresponde a una persona. Algunos puntos y líneas se superponen porque varias personas pueden tener las mismas medidas. Mostraremos primero **todos los pares** y después, en otra celda, solo los que difieren tras redondear.

In [ ]:
def graficos_apareados(tabla, titulo='Todos los pares'):  # Define una función que representa varias vistas de datos emparejados.
    """Relación, Bland–Altman, diferencias individuales y perfiles por par."""  # Describe los gráficos que genera la función.
    par = tabla[['LftSpan', 'RtSpan']].dropna()  # Selecciona medidas izquierda/derecha y descarta pares incompletos.
    x, y = par['LftSpan'].to_numpy(), par['RtSpan'].to_numpy()  # Convierte cada medida emparejada a arreglos NumPy.
    diff, promedio = y - x, (x + y) / 2  # Calcula derecha−izquierda y el promedio de cada par.
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))  # Crea una cuadrícula de cuatro gráficos.

    axes[0, 0].scatter(x, y, s=65, color='#9fc3e4', edgecolor='black', linewidth=0.8)  # Grafica cada par de medidas.
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())  # Busca extremos comunes para la escala de ambos ejes.
    axes[0, 0].plot([lo, hi], [lo, hi], color='#800000', linewidth=1.8)  # Añade la referencia de igualdad y=x.
    axes[0, 0].set(xlabel='Izquierda (cm)', ylabel='Derecha (cm)', title='Comparación de medidas')  # Etiqueta y titula la relación.

    axes[0, 1].scatter(promedio, diff, s=65, color='#9fc3e4', edgecolor='black', linewidth=0.8)  # Dibuja el Bland–Altman: promedio frente a diferencia.
    media_diff = diff.mean()  # Calcula la diferencia media, o sesgo promedio.
    desviacion_diff = diff.std(ddof=1)  # Calcula la desviación típica muestral de las diferencias.
    axes[0, 1].axhline(media_diff, color='#800000', linewidth=2, label='Diferencia media')  # Marca el sesgo medio con una línea horizontal.
    for nivel in (media_diff - 1.96 * desviacion_diff, media_diff + 1.96 * desviacion_diff):  # Calcula los límites descriptivos de acuerdo.
        axes[0, 1].axhline(nivel, color='#800000', ls='--', linewidth=1.6)  # Dibuja cada límite de acuerdo como línea discontinua.
    axes[0, 1].set(xlabel='Media de cada par (cm)', ylabel='Derecha − izquierda (cm)', title='Bland–Altman')  # Etiqueta y titula el gráfico.

    axes[1, 0].scatter(np.arange(1, len(diff) + 1), diff, s=65, color='#9fc3e4',  # Grafica cada diferencia individual frente a su posición.
                       edgecolor='black', linewidth=0.8)  # Añade borde a los puntos para hacerlos visibles.
    axes[1, 0].axhline(0, color='#800000', ls='--', linewidth=1.8)  # Añade cero como referencia de igualdad entre medidas.
    axes[1, 0].set(xlabel='Posición del par', ylabel='Derecha − izquierda (cm)', title='Diferencia por persona')  # Etiqueta y titula las diferencias individuales.

    ax_perfiles = axes[1, 1]  # Guarda el eje inferior derecho para los perfiles individuales.
    etiquetas_vistas = set()  # Registra categorías ya etiquetadas para evitar leyendas repetidas.
    for izquierda, derecha in zip(x, y):  # Recorre las dos medidas de cada persona conservando los pares.
        if derecha < izquierda:  # Comprueba si la medida derecha es menor que la izquierda.
            color, etiqueta, transparencia = '#34699a', 'Derecha menor', 0.45  # Asigna estilo para diferencias negativas.
        elif derecha > izquierda:  # Comprueba si la medida derecha supera a la izquierda.
            color, etiqueta, transparencia = '#9fc3e4', 'Derecha mayor', 0.65  # Asigna estilo para diferencias positivas.
        else:  # Este caso corresponde a medidas iguales.
            color, etiqueta, transparencia = '#b0b0b0', 'Iguales', 0.25  # Asigna estilo neutro para igualdad.
        # Dos puntos (uno por palma) unidos por la línea de esa persona.
        ax_perfiles.plot([0, 1], [izquierda, derecha], color=color, alpha=transparencia,  # Une las dos medidas de la persona con el color elegido.
                         linewidth=1.1, marker='o', markersize=4,  # Ajusta línea y marcadores en cada perfil.
                         markeredgecolor='#444444', markeredgewidth=0.5,  # Define el borde de los marcadores.
                         label=etiqueta if etiqueta not in etiquetas_vistas else None)  # Añade una sola entrada de leyenda por categoría.
        etiquetas_vistas.add(etiqueta)  # Marca la categoría como ya incluida en la leyenda.
    ax_perfiles.plot([0, 1], [x.mean(), y.mean()], color='#800000', linewidth=3,  # Une las medias de las dos manos como referencia global.
                     marker='o', markersize=8, label='Media de cada lado', zorder=5)  # Destaca las medias y las coloca delante de los perfiles.
    ax_perfiles.set(xticks=[0, 1], xticklabels=['Izquierda', 'Derecha'],  # Etiqueta las dos posiciones del eje horizontal.
                    ylabel='Palma (cm)', title='Perfiles de cada persona')  # Etiqueta la magnitud y titula el gráfico.
    ax_perfiles.set_xlim(-0.1, 1.1)  # Añade un pequeño margen a los extremos del eje X.
    ax_perfiles.legend(loc='lower right', fontsize=10)  # Muestra la leyenda sin ocultar demasiado los perfiles.

    for eje in axes.flat:  # Recorre los cuatro ejes creados.
        eje.title.set_fontsize(18)  # Ajusta el tamaño del título de cada panel.
        eje.xaxis.label.set_fontsize(16)  # Ajusta el tamaño de la etiqueta horizontal.
        eje.yaxis.label.set_fontsize(16)  # Ajusta el tamaño de la etiqueta vertical.
        eje.tick_params(labelsize=14)  # Ajusta el tamaño de las marcas de escala.
    fig.suptitle(titulo, fontsize=20)  # Añade un título general a la figura.
    fig.tight_layout()  # Organiza los paneles para evitar solapamientos.
    plt.show()  # Muestra la figura con los cuatro gráficos.

#### Fase 1: todos los pares completos

Dibujamos las cuatro vistas sin filtrar por el tamaño ni el signo de la diferencia. Las líneas horizontales grises en el gráfico de perfiles indican igualdad exacta entre ambas palmas; las inclinadas permiten identificar qué lado es mayor. En el gráfico de diferencias, el eje horizontal indica la **posición de cada par** en los datos utilizados, no el identificador original de la persona.

In [ ]:
graficos_apareados(PennState1)  # Genera los cuatro gráficos usando todos los pares completos de palmas.

#### Fase 2: pares con longitudes redondeadas distintas

Volvemos a dibujar las cuatro vistas, pero excluimos los pares cuyas longitudes derecha e izquierda quedan **iguales al redondearlas al entero**. Antes descartamos los pares incompletos. Esto reduce la superposición de líneas horizontales y facilita ver las diferencias individuales.

**Importante:** redondear y filtrar es solo un recurso visual, no equivale a exigir una diferencia mínima fija ni justifica un nuevo contraste sobre los pares seleccionados. La prueba apareada que aparece **a continuación** utiliza todos los pares completos.

In [ ]:
pares_redondeados = PennState1.dropna(subset=['LftSpan', 'RtSpan'])  # Mantiene solo personas con ambas medidas conocidas.
distintos_redondeados = pares_redondeados['RtSpan'].round() != pares_redondeados['LftSpan'].round()  # Marca pares cuyas medidas redondeadas difieren.
PennState1_sub = pares_redondeados.loc[distintos_redondeados]  # Crea el subconjunto solo para mejorar la visualización.
print(f'Pares seleccionados: {len(PennState1_sub)} de {len(pares_redondeados)}')  # Informa cuántos pares se muestran frente al total completo.
graficos_apareados(PennState1_sub, 'Pares con longitudes redondeadas distintas')  # Dibuja los gráficos para el subconjunto seleccionado.

#### Prueba t para muestras apareadas: comparación correcta

Tras examinar los gráficos, contrastamos la **diferencia media con todos los pares completos**, no solo con los 62 seleccionados para la segunda visualización. Cada persona aporta un par de valores. Definimos $D = \mathrm{RtSpan} - \mathrm{LftSpan}$ (derecha menos izquierda) y contrastamos **si la diferencia media poblacional es cero**:

$$
\left\{
\begin{aligned}
H_0 &: \mu_D = 0 \\
H_1 &: \mu_D \ne 0
\end{aligned}
\right.
$$

`stats.ttest_rel` calcula la diferencia **dentro de cada persona** y aplica una prueba t de una muestra a esas diferencias. El estadístico t mide cuántos errores estándar separan la diferencia media observada de cero; el valor p bilateral considera diferencias extremas en ambos sentidos si $H_0$ fuese cierta. Si $p<0{,}05$, rechazamos $H_0$; si no, no la rechazamos (eso no demuestra que las palmas sean idénticas). Suponemos que los pares de distintas personas son independientes y que las **diferencias** son aproximadamente normales, sobre todo si la muestra es pequeña. El signo de la diferencia media indica qué palma es mayor en promedio.

In [ ]:
# Restamos las medidas de cada persona en el mismo orden que en la prueba.
diferencias_palma = pares['RtSpan'] - pares['LftSpan']  # Calcula, para cada persona, derecha menos izquierda.
resultado_apareado = stats.ttest_rel(pares['RtSpan'], pares['LftSpan'])  # Ejecuta la prueba t apareada sobre las dos mediciones emparejadas.
print(f'Pares: {len(pares)}; diferencia media (derecha − izquierda): {diferencias_palma.mean():.2f} cm')  # Informa el número de personas y la diferencia media.
print(f't = {resultado_apareado.statistic:.4f}; p-valor bilateral = {resultado_apareado.pvalue:.4g}')  # Muestra el estadístico t y el p-valor bilateral.
if resultado_apareado.pvalue < 0.05:  # Compara el p-valor con el nivel de significación del 5 %.
    print('Se rechaza H₀ al nivel del 5 %: hay evidencia de una diferencia media entre las palmas.')  # Comunica la decisión cuando el resultado es significativo.
else:  # Se ejecuta cuando el p-valor es al menos 0,05.
    print('No se rechaza H₀ al nivel del 5 %: no hay evidencia suficiente de una diferencia media.')  # Comunica que no hay evidencia suficiente contra H₀.

#### Microejercicio: intervalo para una diferencia apareada

Partimos de `diferencias_palma`, definida como derecha menos izquierda para cada persona. Calcula un IC del 99 % para la diferencia media con una prueba t de una muestra sobre esas diferencias. Completa el nivel de confianza e interpreta si el intervalo incluye 0 y qué significa el signo.

In [ ]:
try:
    # Microejercicio: calcula un intervalo para la media de las diferencias emparejadas.
    resultado_palma_99 = stats.ttest_1samp(diferencias_palma, popmean=...)  # Completa el valor nulo y contrasta si la diferencia media es cero.
    ic_palma_99 = resultado_palma_99.confidence_interval(confidence_level=...)  # Completa el nivel de confianza requerido.
    print(f'IC del 99 % para la diferencia media (derecha − izquierda): ({ic_palma_99.low:.3f}, {ic_palma_99.high:.3f})')  # Imprime los extremos del intervalo.
    # Comprueba si el intervalo incluye 0 y relaciona la conclusión con la prueba apareada.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Ejercicio 2.2

1. Crea una variable `origin2` que valga `JFK` si el origen es ese aeropuerto, y `Other` en los demás casos; puedes usar `np.where`.
2. Compara las medias del retraso en las llegadas entre `JFK` y el resto.
3. Compara los retrasos en la salida y la llegada de **los mismos vuelos**. Obtén un IC del **99,9 %** para la diferencia de medias.
4. Dibuja un gráfico de Bland–Altman para los **10 000 primeros** vuelos con retraso en la llegada inferior a 180 minutos. Utiliza las columnas de retraso en salida y llegada y la función de gráficos apareados anterior, o adapta su cálculo de medias y diferencias a esas columnas.

#### Lectura de los datos de vuelos

Cargamos aquí el fichero necesario para el ejercicio 2.2. La tabla `vuelos_df` se reutiliza en los ejercicios 2.3 y 2.4; se lee una vez, justo antes de comenzar estos ejercicios. El fichero está separado por tabulaciones y `@`, `NA` y las celdas vacías se interpretan como valores ausentes.

In [ ]:
vuelos_df = pd.read_csv(DATA_PATH / 'flights.txt', sep='\t', na_values=['@', 'NA', ''])  # Lee vuelos_df desde el archivo tabulado y reconoce marcadores de datos ausentes.
display(vuelos_df.head())  # Muestra las primeras filas como comprobación de la carga.

In [ ]:
try:
    # 2.2.1 — Clasificación del aeropuerto de salida.
    # Usa loc y vuelos_df['origin2'] para separar JFK y otros aeropuertos; compara sus frecuencias.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.2.2 — Compara los retrasos medios de llegada entre JFK y Other.
    # Separa vuelos_df['arr_delay'] por origin2 con loc y dropna(); resume grupos y aplica Welch.
    # Calcula el IC bilateral y conserva el orden JFK − Other al interpretarlo.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.2.3 — IC del 99,9 % para la diferencia apareada llegada − salida.
    # Selecciona los pares completos, calcula arr_delay − dep_delay y reutiliza el patrón t apareado.
    # Interpreta el intervalo en minutos y comprueba si contiene cero.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.2.4 — Gráfico de Bland–Altman para vuelos.
    # Usa los pares completos de arr_delay y dep_delay; representa la diferencia frente a su promedio.
    # Añade la diferencia media y límites descriptivos de acuerdo como en graficos_apareados(...).
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Comparación de dos varianzas (opcional)

Retomamos las facturas válidas de `Plan` y `Prepaid` preparadas anteriormente. Primero exploramos gráficamente la **dispersión**; después comparamos las **varianzas poblacionales** con una prueba F y calculamos, por separado, intervalos de confianza para el cociente de varianzas y el cociente de desviaciones típicas, siempre en el orden **Plan / Prepaid**.

La prueba F y sus intervalos requieren observaciones independientes en ambos grupos y poblaciones aproximadamente **normales**. La asimetría y los valores atípicos de las facturas aconsejan interpretar los resultados con cautela.

### Exploración gráfica de la dispersión

Dibujamos las facturas de `Plan` y `Prepaid` en **dos cajas sobre la misma escala**. Una caja más alta indica mayor amplitud entre los cuartiles; también conviene comparar los bigotes y los valores atípicos. Estas diferencias visuales son descriptivas y no bastan por sí mismas para afirmar que las varianzas poblacionales difieran.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura amplia y un eje para el boxplot.
ax.boxplot([grupo1, grupo2], tick_labels=[nombre1, nombre2], patch_artist=True,  # Dibuja cajas para los dos grupos con etiquetas.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Define relleno y borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Da estilo a los bigotes y a sus extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Resalta las medianas.
ax.set_title('Facturas por tipo de contrato: comparación de la dispersión', fontsize=20)  # Añade título al gráfico.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta la variable de respuesta.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de los valores de los ejes.
plt.show()  # Muestra la comparación de dispersión.

### Función para comparar la variabilidad

Definimos `prueba_varianzas(a, b, confianza=0.95)` para reutilizarla en otros datos, como en el ejercicio siguiente. La función elimina los ausentes de cada grupo, calcula el estadístico y el valor p de un **contraste F bilateral**, y devuelve dos intervalos de confianza: uno para $\sigma_a^2/\sigma_b^2$ y otro para $\sigma_a/\sigma_b$. El **orden de los argumentos importa**: aquí `a` es `Plan` y `b` es `Prepaid`. A continuación examinamos sus resultados paso a paso.

In [ ]:
def prueba_varianzas(a, b, confianza=0.95):  # Define una función para comparar la dispersión de dos muestras independientes.
    """Contraste F bilateral e IC del cociente de varianzas y desviaciones."""  # Resume los resultados que devuelve la función.
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()  # Convierte ambos argumentos en series y elimina sus valores ausentes.
    gl1, gl2 = len(a)-1, len(b)-1  # Calcula los grados de libertad de numerador y denominador.
    cociente = a.var(ddof=1) / b.var(ddof=1)  # Calcula el cociente de varianzas muestrales en el orden a/b.
    pvalor = min(1, 2 * min(stats.f.cdf(cociente, gl1, gl2), stats.f.sf(cociente, gl1, gl2)))  # Calcula el p-valor bilateral usando la cola menor.
    alfa = 1-confianza  # Convierte el nivel de confianza en el nivel de significación.
    ic_var = (cociente/stats.f.ppf(1-alfa/2, gl1, gl2),  # Calcula el límite inferior del IC del cociente de varianzas.
              cociente/stats.f.ppf(alfa/2, gl1, gl2))  # Calcula el límite superior del intervalo.
    return {'F': cociente, 'p-valor': pvalor, 'IC varianzas': ic_var,  # Devuelve el cociente F, el p-valor y el IC de varianzas.
            'IC desviaciones': tuple(np.sqrt(ic_var))}  # Devuelve el IC del cociente de desviaciones típicas aplicando raíces.

#### Contraste F bilateral: ¿son iguales las varianzas?

Comparamos las **varianzas poblacionales** de las facturas con las hipótesis:

$$
\left\{
\begin{aligned}
H_0 &: \sigma^2_{\mathrm{Plan}} = \sigma^2_{\mathrm{Prepaid}} \\
H_1 &: \sigma^2_{\mathrm{Plan}} \ne \sigma^2_{\mathrm{Prepaid}}
\end{aligned}
\right.
$$

El estadístico $F=s^2_{\mathrm{Plan}}/s^2_{\mathrm{Prepaid}}$ sería próximo a 1 si ambas dispersiones fueran parecidas. La prueba calcula un **valor p bilateral** con la distribución F y $n_{\mathrm{Plan}}-1$ y $n_{\mathrm{Prepaid}}-1$ grados de libertad. Si $p<0{,}05$, rechazamos $H_0$; en otro caso no la rechazamos, **sin demostrar** que las varianzas sean exactamente iguales. Esta prueba es sensible a desviaciones de la normalidad.

In [ ]:
resultado_varianzas = prueba_varianzas(grupo1, grupo2)  # Ejecuta el contraste F y calcula intervalos para Plan/Prepaid.
print(f'Contraste F bilateral ({nombre1} / {nombre2}): '  # Identifica el orden del cociente que se está informando.
      f'F = {resultado_varianzas["F"]:.4f}; p-valor = {resultado_varianzas["p-valor"]:.4g}')  # Muestra el estadístico y el p-valor.
if resultado_varianzas['p-valor'] < 0.05:  # Compara el p-valor con alfa = 0,05.
    print('Se rechaza la igualdad de varianzas al nivel del 5 %.')  # Informa de evidencia de varianzas distintas.
else:  # Se ejecuta si el p-valor es al menos 0,05.
    print('No se rechaza la igualdad de varianzas al nivel del 5 %.')  # Informa que no hay evidencia suficiente para rechazar igualdad.

#### IC del 95 % para el cociente de varianzas

El contraste anterior decide si hay evidencia contra la igualdad; el **intervalo de confianza** estima cuánto podría valer $\sigma^2_{\mathrm{Plan}}/\sigma^2_{\mathrm{Prepaid}}$. El cociente muestral $F$ se divide entre cuantiles de la distribución F con los grados de libertad de ambos grupos. Con `confianza=0.95`, el intervalo es del **95 %**; si se aumenta a 0,99, el intervalo suele ser más ancho. El valor **1** representa varianzas iguales: un IC que no lo incluya es coherente con rechazar la igualdad en un contraste bilateral al 5 %. Este intervalo también requiere normalidad e independencia.

In [ ]:
ic_var = resultado_varianzas['IC varianzas']  # Recupera el intervalo para el cociente de varianzas.
print(f'IC del 95 % para σ² de {nombre1} / σ² de {nombre2}: '  # Indica qué cociente y nivel de confianza se muestran.
      f'({ic_var[0]:.2f}, {ic_var[1]:.2f})')  # Completa e imprime los dos límites.

#### IC del 95 % para el cociente de desviaciones típicas

La desviación típica es la **raíz cuadrada de la varianza**. Por eso obtenemos este intervalo aplicando la raíz cuadrada a los dos límites del IC anterior: ahora estimamos $\sigma_{\mathrm{Plan}}/\sigma_{\mathrm{Prepaid}}$, no el cociente de varianzas. Un valor de **1** indica igual desviación típica; un intervalo situado completamente por encima de 1 apunta a mayor dispersión en `Plan`. El nivel de confianza y los supuestos de normalidad e independencia son los mismos que antes.

In [ ]:
ic_desv = resultado_varianzas['IC desviaciones']  # Recupera el intervalo del cociente de desviaciones típicas.
print(f'IC del 95 % para σ de {nombre1} / σ de {nombre2}: '  # Identifica el parámetro y el orden del cociente.
      f'({ic_desv[0]:.2f}, {ic_desv[1]:.2f})')  # Muestra los límites con dos decimales.

#### Microejercicio: repetir la comparación de varianzas al 99 %

Usa `prueba_varianzas` con los mismos grupos `Plan` y `Prepaid`, pero eleva la confianza al 99 %. Completa el nivel y extrae los dos intervalos del resultado. ¿El intervalo de varianzas contiene 1? Compáralo con el contraste F al 1 %.

In [ ]:
try:
    # Microejercicio: repite el análisis de varianzas con confianza del 99 %.
    resultado_varianzas_99 = prueba_varianzas(grupo1, grupo2, confianza=...)  # Completa el nivel de confianza solicitado.
    ic_var_99 = resultado_varianzas_99['IC varianzas']  # Extrae el intervalo del cociente de varianzas.
    ic_desv_99 = resultado_varianzas_99['IC desviaciones']  # Extrae el intervalo del cociente de desviaciones típicas.
    print(f'F = {resultado_varianzas_99["F"]:.3f}; p = {resultado_varianzas_99["p-valor"]:.4g}')  # Muestra estadístico y p-valor.
    print(f'IC 99 % varianzas: ({ic_var_99[0]:.3f}, {ic_var_99[1]:.3f})')  # Imprime los límites de varianza.
    print(f'IC 99 % desviaciones: ({ic_desv_99[0]:.3f}, {ic_desv_99[1]:.3f})')  # Imprime los límites para desviaciones.
    # Interpreta si el cociente 1 es compatible con los datos.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Ejercicio 2.3

1. Representa diagramas de cajas del retraso en la llegada según si el aeropuerto de salida es `JFK` o no.
2. Repite el diagrama de cajas **solo** para los retrasos positivos y utiliza escala vertical logarítmica (`plt.yscale('log')`).
3. Compara las varianzas y calcula un IC del **99 %** con `prueba_varianzas(..., confianza=0.99)`.

In [ ]:
try:
    # 2.3.1 — Boxplots del retraso de llegada según origen.
    # Usa origin2 y arr_delay para crear los grupos JFK y Other con loc; elimina ausentes con dropna(), conserva ceros y negativos.
    # Dibuja las dos series con ax.boxplot(...) y compara medianas, dispersión y atípicos.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.3.2 — Repite el boxplot solo con retrasos positivos.
    # Usa loc con arr_delay > 0 y vuelve a separar JFK y Other.
    # Dibuja con ax.boxplot(...) y cambia la escala usando ax.set_yscale('log').
    # Explica por qué los ceros y negativos no caben en escala logarítmica y compara las colas.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.3.3 — Compara varianzas e IC del 99 % para JFK y Other.
    # Usa los retrasos originales y los grupos del apartado anterior, eliminando ausentes.
    # Reutiliza prueba_varianzas(..., confianza=0.99); indica el orden del cociente.
    # Examina F, p-valor e intervalos; comenta si el cociente 1 es compatible y relaciona la decisión con alfa=0,01.
    # Advierte sobre independencia, normalidad y sensibilidad de la prueba F a valores atípicos.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Comparación de dos proporciones

Estudiamos si la **proporción de estudiantes con contrato `Prepaid`** es la misma en los dos grupos de `Gender`. Primero construimos una tabla de frecuencias, después visualizamos la distribución conjunta en un mosaico y, por último, realizamos un contraste χ². Todos los pasos utilizan únicamente estudiantes con `Gender` y `PlanType` conocidos.

### Tabla de frecuencias: género y tipo de contrato

La **tabla de contingencia** cuenta cuántos estudiantes tienen cada combinación de `Gender` (filas) y `PlanType` (columnas). `pd.crosstab` excluye las filas con valores ausentes en cualquiera de las dos variables. Para comparar la proporción de `Prepaid` entre géneros, fíjate en los casos de `Prepaid` **respecto al total de cada fila**, no solo en sus recuentos absolutos.

In [ ]:
t1 = pd.crosstab(datos['Gender'], datos['PlanType'])  # Cuenta cada combinación de género y tipo de contrato en una tabla de contingencia.
display(t1)  # Muestra los recuentos observados para comparar las proporciones.

### Mosaico: comparación visual de proporciones

El **mosaico** representa las mismas observaciones válidas que la tabla. La **anchura** de cada bloque de género refleja cuántas personas hay en ese grupo, mientras que la **altura relativa** de `Prepaid` dentro de cada bloque muestra su proporción en ese género. Si ambas alturas fueran similares, las proporciones observadas también lo serían. El área de cada rectángulo refleja la frecuencia conjunta; el gráfico describe los datos, pero no sustituye al contraste.

In [ ]:
from statsmodels.graphics.mosaicplot import mosaic  # Importa la función para representar tablas de contingencia como mosaicos.

fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura y un eje para el gráfico.
mosaic(datos.dropna(subset=['Gender', 'PlanType']), ['Gender', 'PlanType'], ax=ax,  # Grafica las combinaciones de género y contrato, excluyendo filas incompletas.
       properties=lambda grupo: {'facecolor': '#9fc3e4' if grupo[1] == 'Prepaid' else '#6f9fc8',  # Asigna un color distinto según el tipo de plan.
                                 'edgecolor': '#666666', 'linewidth': 1.2})  # Define el borde de los rectángulos.
ax.set_title('Género y tipo de contrato', fontsize=20)  # Añade un título que describe las variables.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas de los ejes.
for etiqueta in ax.texts:  # Recorre los textos generados por la función de mosaico.
    etiqueta.set_fontsize(14)  # Aumenta el tamaño de las etiquetas de las categorías.
plt.show()  # Muestra el mosaico.

### Contraste χ²: ¿es igual la proporción de prepago?

Formulamos un **contraste bilateral de independencia**, equivalente aquí a comparar las proporciones poblacionales de `Prepaid` entre los dos géneros:

$$
\left\{
\begin{aligned}
H_0 &: \pi_{\mathrm{Prepaid|Female}} = \pi_{\mathrm{Prepaid|Male}} \\
H_1 &: \pi_{\mathrm{Prepaid|Female}} \ne \pi_{\mathrm{Prepaid|Male}}
\end{aligned}
\right.
$$

La prueba χ² compara las **frecuencias observadas** de la tabla con las **esperadas si $H_0$ fuera cierta**. Para esta tabla 2 × 2 usamos la corrección de continuidad de Yates (`correction=True`). Con un nivel del 5 %, rechazamos $H_0$ si $p<0{,}05$; de lo contrario, no la rechazamos: **no demostrar una diferencia no equivale a demostrar igualdad**. El contraste supone observaciones independientes y frecuencias esperadas suficientemente grandes; `esperado` permite comprobar estas últimas.

In [ ]:
chi_cuadrado, pvalor, gl, esperado = stats.chi2_contingency(t1, correction=True)  # Calcula χ², su p-valor, grados de libertad y frecuencias esperadas con corrección de Yates.
print(f'Contraste χ² de proporciones (corrección por continuidad): χ² = {chi_cuadrado:.4f}; '  # Inicia el resumen e indica la corrección aplicada.
      f'gl = {gl}; p-valor = {pvalor:.4g}')  # Añade grados de libertad y p-valor al mensaje.
print(f'Frecuencia esperada mínima: {esperado.min():.2f}')  # Muestra la menor frecuencia esperada para revisar el supuesto.
if pvalor < 0.05:  # Comprueba si el p-valor es menor que alfa = 0,05.
    print('Se rechaza la igualdad de proporciones de prepago al nivel del 5 %.')  # Indica que hay evidencia de proporciones distintas.
else:  # Se ejecuta cuando el p-valor es al menos 0,05.
    print('No se rechaza la igualdad de proporciones de prepago al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar igualdad.

#### Microejercicio: proporciones por género

Con la tabla `t1`, calcula las proporciones de `Prepaid` dentro de cada género y vuelve a hacer el contraste χ² sin corrección de continuidad. Completa los argumentos, compara el valor p con el del contraste anterior y explica si cambia la decisión al 5 %.

In [ ]:
try:
    # Microejercicio: compara las proporciones de contratos por género y repite el contraste sin Yates.
    proporciones_genero = t1.div(t1.sum(axis=...), axis=...)  # Completa los ejes para dividir cada recuento entre el total de su fila.
    display(proporciones_genero)  # Muestra las proporciones de cada tipo de contrato dentro de cada género.
    chi_sin_yates, p_sin_yates, gl_sin_yates, esperado_sin_yates = stats.chi2_contingency(t1, correction=...)  # Completa el argumento para activar o quitar la corrección.
    print(f'χ² = {chi_sin_yates:.4f}; p = {p_sin_yates:.4g}; gl = {gl_sin_yates}')  # Imprime estadístico, p-valor y grados de libertad.
    # Compara la decisión con el contraste con corrección de Yates.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Ejercicio 2.4

1. Crea tres variables en `vuelos_df` (vuelos):
   - `date`: fecha completa en formato `YYYY-MM-DD`; construye la fecha con `pd.to_datetime` a partir de año, mes y día.
   - `weekend`: `True` si la fecha cae en sábado o domingo, `False` en caso contrario; puedes usar `.dt.dayofweek >= 5`.
   - `arr_delay_cat`: `True` si el vuelo llega con retraso, `False` en caso contrario. **Ten en cuenta los valores ausentes.**
2. Representa un mosaico del retraso (`arr_delay_cat`) y el fin de semana (`weekend`).
3. Comprueba si la proporción de vuelos **sin retraso** es igual en fines de semana y en el resto de los días mediante una tabla de contingencia y una prueba χ² de proporciones.

In [ ]:
try:
    # 2.4.1 — Crea date, weekend y arr_delay_cat en vuelos_df.
    # Convierte date a fecha, deriva el día de semana y clasifica el retraso como en las instrucciones.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.4.2 — Mosaico de retraso frente a fin de semana.
    # Reutiliza las variables creadas arriba y la función mosaic del ejemplo anterior.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.4.3 — Contraste χ² de la proporción de vuelos sin retraso.
    # Construye la tabla de contingencia con las variables creadas arriba y reutiliza stats.chi2_contingency(...).
    # Interpreta el p-valor y las frecuencias esperadas, teniendo en cuenta las limitaciones del contraste.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Errores de tipo I y tipo II (opcional)

### Simulación del error de tipo I

Estimamos la proporción de resultados significativos **cuando las medias son iguales**. Generamos dos grupos de 100 retrasos normales con media 10 y desviación 3, repetimos la prueba 1 000 veces y calculamos la fracción de valores p menores de 0,05 (error de tipo I, α). Fijamos una semilla para repetir el experimento.

In [ ]:
n = 100  # Define el tamaño muestral de cada grupo simulado.
nsim = 1000  # Establece cuántas veces se repetirá el experimento.
p = []  # Inicializa una lista para guardar los p-valores de cada simulación.
rng = np.random.default_rng(12345)  # Fija una semilla para poder reproducir las simulaciones.
for i in range(nsim):  # Repite el experimento el número de veces indicado.
    RetrasosPrat = rng.normal(loc=10, scale=3, size=n)  # Genera el primer grupo normal con media 10 y desviación típica 3.
    RetrasosBarajas = rng.normal(loc=10, scale=3, size=n)  # Genera el segundo grupo bajo la misma media nula.
    tt = stats.ttest_ind(RetrasosPrat, RetrasosBarajas, equal_var=True)  # Aplica la prueba t independiente con varianzas iguales.
    p.append(tt.pvalue)  # Guarda el p-valor para esta repetición.
print('Error tipo I estimado (α):', np.mean(np.array(p) < 0.05))  # Estima la fracción de falsos rechazos cuando H₀ es cierta.

### Simulación del error de tipo II (β)

El error de tipo II es la proporción de resultados **no significativos cuando sí existe una diferencia**.

### Ejercicio 2.5

1. Por simulación, calcula β con **393 observaciones por grupo**, una diferencia de medias de **0,2** y varianza **1** en ambos grupos. Elige `n`, `nsim` y una lista de valores p; fija una semilla y, en cada simulación, genera los dos grupos normales, aplica la prueba t de muestras independientes con varianzas iguales y guarda el valor p. Calcula la proporción de valores p **mayores o iguales a 0,05**.
2. La potencia de un análisis es **1 − β**. Calcula también la potencia teórica con `statsmodels.stats.power.TTestIndPower().power(effect_size=0.2, nobs1=393, alpha=0.05, ratio=1)` y comprueba si se aproxima a la simulación.

In [ ]:
try:
    # Plantilla del ejercicio 2.5: completa los puntos suspensivos y ejecuta la celda.
    # Descomenta y completa el código de simulación de la potencia que aparece en esta plantilla.
    # Añade o adapta el código dentro de este bloque, con sangría.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Reto final integrador (opcional)

Compara los retrasos de llegada entre vuelos que salen de JFK y del resto de aeropuertos. Esta vez decide qué contraste de medias corresponde si no quieres suponer varianzas iguales; calcula también un IC del 95 % para la diferencia JFK − Other y usa `prueba_varianzas` como comprobación descriptiva adicional. Termina con una conclusión que integre tamaños muestrales, medias, intervalo, valor p y cautelas: los retrasos son asimétricos y la prueba F presupone normalidad.

**Pista:** reutiliza la clasificación `origin2`, los retrasos válidos y las funciones vistas en los apartados anteriores. Mantén el mismo orden de grupos en el contraste y en el intervalo.

In [ ]:
try:
    # Reto final integrador (opcional): escribe el análisis desde cero.
    # Usa vuelos_df para construir los grupos JFK y Other de arr_delay con loc y dropna().
    # Resume tamaños y medias; usa stats.ttest_ind(..., equal_var=False) para Welch.
    # Obtén el IC bilateral del 95 % con confidence_interval() y mantén el orden JFK − Other.
    # Reutiliza prueba_varianzas(...) como comparación complementaria, indicando que arr_delay no es normal.
    # Cierra con una conclusión que integre estimación, p-valor, supuestos y relevancia práctica.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El reto no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')